<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
DeepAgents: Parameter, Sandbox & Einordnung
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** M32 hat den DeepAgents-Kern gezeigt (Planning, Tools, Sub-Agenten). Dieses Notebook vertieft die Betriebsseite: weitere Steuerparameter, isolierte Remote-Sandboxes für Code-Ausführung, und die Einordnung — wann DeepAgents statt manuellem LangGraph sinnvoll ist.

> **Voraussetzung:** M32 — DeepAgents-Kern (`ki_glossar`, `kurs_modul_info`, `sub_researcher`, `sub_module_expert`).

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q deepagents==0.6.12

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M33-DeepAgents-Vertiefung"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()
 
# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M33_DeepAgents_Vertiefung",
    "tags": ["m33", "deepagents"],
    "metadata": {"notebook": "M33", "version": "1.0"}
}

# 1 | Übersicht
---

**M32** hat einen Koordinator mit zwei Sub-Agenten gebaut (`recherche-spezialist`, `kurs-experte`). Dieses Notebook nutzt dieselben Tools und Sub-Agenten weiter, um zusätzliche `create_deep_agent()`-Parameter, Remote-Sandboxes und die Abgrenzung zu LangGraph zu zeigen.

Dieses Modul setzt **M32_DeepAgents_Harness** voraus: Der DeepAgents-Kern ist bekannt; hier geht es um optionale Parameter, Sandboxen und Einordnung.


In [ ]:
# Setup: Fortsetzung aus M32 (Tools + Sub-Agenten erneut verfügbar machen)
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from IPython.display import Image, display

@tool
def ki_glossar(begriff: str) -> str:
    """Schlägt einen KI-Begriff im Glossar nach.

    Args:
        begriff: Der nachzuschlagende Begriff (z.B. LangGraph, RAG, HITL)
    """
    glossar = {
        "langgraph":   "LangGraph: Zustandsbasierte Multi-Agent-Graphen auf Basis von LangChain. StateGraph, Checkpointing, Streaming.",
        "langchain":   "LangChain: Framework für LLM-Anwendungen. Chains, Agents, LCEL, Tools, RAG.",
        "langsmith":   "LangSmith: Tracing, Evaluation und Monitoring für LLM-Pipelines.",
        "rag":         "RAG (Retrieval-Augmented Generation): Kombination von Vektordatenbank-Suche und LLM-Synthese.",
        "hitl":        "HITL (Human-in-the-Loop): Menschliche Freigabe bei kritischen Agenten-Schritten. LangGraph: interrupt().",
        "harness":     "Harness: Unterstützendes Rahmengerüst für einen Agenten — liefert Planning, Filesystem, Sub-Agenten als Infrastruktur.",
        "deepagents":  "DeepAgents: Offizielles Agent-Harness von LangChain. Batteries Included: Planning, Filesystem, Sub-Agenten, Memory.",
    }
    key = begriff.lower().strip()
    return glossar.get(key, f'Begriff "{begriff}" nicht im Glossar. Verfügbar: {", ".join(glossar.keys())}')

@tool
def kurs_modul_info(modul_nr: str) -> str:
    """Gibt Informationen zu einem Kursmodul zurück.

    Args:
        modul_nr: Modul-Nummer (z.B. M33, M33, M33)
    """
    module = {
        "M08": "StateGraph Basics: State, Nodes, Edges, compile()",
        "M09": "Conditional Routing und Tool-Loop",
        "M17": "Human-in-the-Loop: interrupt(), Approval-Pattern",
        "M18": "Memory-Systeme: Kurz- und Langzeitgedächtnis, Checkpointing",
        "M19": "Multi-Agent Patterns: Parallele Ausführung, Tool-Routing",
        "M20": "Supervisor Pattern: Worker-Agents, Supervisor-Logik, Graph",
        "M21": "Hierarchical Pattern: 3-Ebenen-Delegation, asyncio",
        "M33": "DeepAgents: Autonomes Harness-Pattern (dieses Modul)",
        "M35": "Production Deployment: Deployment, API, Monitoring",
    }
    key = modul_nr.upper().strip()
    return module.get(key, f'Modul "{modul_nr}" nicht gefunden. Verfügbar: {", ".join(module.keys())}')

sub_researcher = {
    "name": "recherche-spezialist",
    "description": "Schlägt KI-Begriffe im Glossar nach und gibt strukturierte Zusammenfassungen zurück",
    "system_prompt": (
        "Du bist Recherche-Spezialist. "
        "Schlage alle relevanten Begriffe im Glossar nach und "
        "gib eine strukturierte Zusammenfassung zurück. "
        "Nutze ausschließlich das ki_glossar-Tool — keine Filesystem-Suche. "
        "Antworte auf Deutsch."
    ),
    "tools": [ki_glossar],
}

sub_module_expert = {
    "name": "kurs-experte",
    "description": "Gibt Informationen zu Kursmodulen und deren Inhalten zurück",
    "system_prompt": (
        "Du bist Kurs-Experte. "
        "Gib Informationen zu Kursmodulen über das kurs_modul_info-Tool zurück — "
        "keine Filesystem-Suche. Antworte auf Deutsch."
    ),
    "tools": [kurs_modul_info],
}

# 2 | Optionale Parameter & Sandbox-Backends
---

<p><font color='black' size="5">
Optional: Weitere Parameter
</font></p>

`create_deep_agent()` bietet weitere praxisrelevante Parameter. Die folgenden Parameter sind Referenzmaterial; der Kern dieses Moduls bleibt Planning, Filesystem und Sub-Agenten:

| Parameter | Beschreibung |
|-----------|-------------|
| `interrupt_on` | HITL-Freigabe vor bestimmten Tool-Calls; **seit 0.6.8** auch für Filesystem-Permissions |
| `memory` | Kontext-Dateien (AGENTS.md) beim Agent-Start automatisch laden |
| `skills` | SKILL.md-Dateien für den Hauptagenten laden (→ M34/M35) |
| `permissions` | Filesystem-Zugriffsrechte einschränken — `list[FilesystemPermission]` (**seit 0.5.2**) |
| `response_format` | Strukturierte Agent-Ausgabe — Pydantic-Modell oder `dict` (**seit 0.5.1**) |
| `name` | Agent-Name für LangSmith-Tracing |
| `cache` | Response-Caching — `BaseCache`-Instanz |
| `backend` | Filesystem-Backend — In-Memory, Local, LangGraph Store oder Sandbox |
| `store` | LangGraph Memory Store für thread-übergreifende Persistenz |
| `context_schema` | Pydantic-Schema für den geteilten Agenten-Kontext |
| `state_schema` | Eigenes State-Schema übergeben (**neu seit 0.6.6**) |
| `debug` | Debug-Ausgaben aktivieren (`True` für detailliertes Logging) |

> ✅ **Sub-Agent-Modell (seit 0.4.11):** Jeder Sub-Agent kann ein eigenes Modell über den `model=`-Schlüssel
> im Dict erhalten. `create_deep_agent()` selbst hat keinen `subagent_model`-Parameter —
> das Modell wird pro Sub-Agenten-Dict gesetzt:
> ```python
> research_subagent = {
>     "name": "recherche",
>     "description": "...",
>     "system_prompt": "...",
>     "model": "openai:gpt-5.6-luna",   # ← eigenes Modell seit 0.4.11
>     "tools": [mein_tool],
> }
> ```

> 🆕 **Neu in 0.6.0:**
> - `CodeInterpreterMiddleware` (experimentell) — JavaScript im Browser-Sandbox ausführen (`deepagents[quickjs]`)
> - `stream_events` v3 — optimiertes Event-Streaming für Langläufer-Agenten

> 🆕 **Neu in 0.6.6–0.6.8:**
> - `state_schema` — eigenes State-Schema übergeben (0.6.6)
> - `DeepAgentState` — jetzt direkt aus `deepagents` importierbar (0.6.7)
> - `interrupt_on` greift jetzt auch für Filesystem-Permissions (0.6.8)

In [ ]:
from genai_lib.model_config import PLANNER
# subagents und tools aus M33 weiterverwenden (siehe Setup-Zelle oben)

extended_agent = create_deep_agent(
    model=init_chat_model(PLANNER),
    tools=[ki_glossar, kurs_modul_info],
    subagents=[sub_researcher, sub_module_expert],
    # HITL — write_file und execute erfordern menschliche Freigabe
    # Seit 0.6.8: interrupt_on greift auch für Filesystem-Permissions
    interrupt_on={"write_file": True, "execute": True},
    # Kontext-Dateien beim Start laden (AGENTS.md-Konvention)
    # memory=["./AGENTS.md"],  # Kommentiert: Datei muss existieren
    system_prompt=(
        "Du bist Koordinator. "
        "Nutze Sub-Agenten für Recherche und Modul-Infos. "
        "Antworte auf Deutsch."
    ),
    checkpointer=InMemorySaver(),
)

**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

✅ **Extended Agent** erstellt mit:

| Parameter | Wert |
|-----------|------|
| `interrupt_on` | `write_file=True`, `execute=True` — HITL aktiv |
| `memory` | auskommentiert (Pfad muss existieren) |

<p><font color='black' size="5">
Optional: Sandbox Backends
</font></p>

Für die Kern-Demo sind keine Remote-Sandboxes nötig. DeepAgents kann Code in **isolierten Remote-Sandboxes** ausführen — ohne Zugriff auf lokale Dateien, Credentials oder das Host-System.

| Backend | Voraussetzung | Typischer Einsatz |
|---------|---------------|-------------------|
| **Modal** | `modal setup` (CLI) | Serverless Python, kostengünstig |
| **Daytona** | `DAYTONA_API_KEY` | Managed Dev-Environments |
| **Runloop** | `RUNLOOP_API_KEY` | Enterprise Sandboxes |
| **LangSmith Sandbox** | LangSmith Account | Integriertes Tracing |

**Empfohlenes Pattern: "Remote Tool"**

> Agent läuft lokal (API-Keys sicher), Code-Ausführung passiert im Sandbox.
> Kosten entstehen nur für die reine Execution-Zeit.

<p><font color='darkblue' size="4">🚨 <b>Achtung</b></font></p>

Für die Demos in diesem Notebook wird kein Sandbox benötigt. Der folgende Code ist auskommentiert und dient als Referenz-Template.

In [ ]:
from genai_lib.model_config import CODING
# ⚠️ Auskommentiert: Modal/Daytona/Runloop Account + API-Key erforderlich
# Dieses Template zeigt die Struktur — zum Ausführen Account aktivieren

# ── Modal Backend ────────────────────────────────────────────────────────
# import modal
# from deepagents.backends import ModalSandbox
#
# app = modal.App.lookup("deepagents-demo", create_if_missing=True)
# sb  = modal.Sandbox.create(app=app)
# backend = ModalSandbox(sandbox=sb)
#
# sandbox_agent = create_deep_agent(
#     model=init_chat_model(CODING),
#     backend=backend,   # ← Code läuft im Sandbox, nicht lokal
#     system_prompt="Schreibe Python-Code und führe ihn aus. Antworte auf Deutsch.",
# )
# result = sandbox_agent.invoke({"messages": [
#     {"role": "user", "content": "Berechne Fibonacci bis 10 und zeige den Code."}
# ]}, config=run_cfg)


In [ ]:
#@markdown   <p><font size="4" color='green'>  Architektur: Remote Tool Pattern</font> </br></p>

# ── Architektur: Remote Tool Pattern ─────────────────────────────────────
diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph lokal ["Lokaler Agent (API-Keys sicher)"]
        A["create_deep_agent(\nbackend=sandbox)"]
    end
    subgraph sandbox ["Remote Sandbox (Modal / Daytona / Runloop)"]
        C["execute(code)"]
        D["ls / read_file\nwrite_file / edit_file"]
        E[("Isoliertes\nFilesystem")]
    end
    A -->|"Tool Call"| C
    A -->|"Tool Call"| D
    C --- E
    D --- E
    C -->|"stdout / stderr"| A
    D -->|"Datei-Inhalt"| A
'''
mermaid(diagram, width=650)
mprint("**Remote Tool Pattern:** Agent lokal · Code-Ausführung im Sandbox · API-Keys nie im Sandbox")

# 3 | Optionaler Deep Dive: Vergleich & Einordnung
---


<p><font color='black' size="5">
Wann LangGraph manuell, wann DeepAgents?
</font></p>

DeepAgents ist kein Ersatz für LangGraph — es ist eine Schicht darüber.
Wähle das richtige Werkzeug je nach Anforderung:

| Kriterium | LangGraph manuell | DeepAgents Harness |
|-----------|-------------------|--------------------|
| **Transparenz** | ✅ Vollständig (jeder Node nachvollziehbar) | ⚠️ Gekapselt (Harness entscheidet) |
| **Kontrolle** | ✅ Präzise (Custom Routing, Custom State) | ⚠️ Opinionated (Harness-Defaults) |
| **Code-Aufwand** | ⚠️ Höher (State, Nodes, Routing explizit) | ✅ Minimal (`create_deep_agent()`) |
| **Planning** | ⚠️ Selbst bauen | ✅ `write_todos` automatisch |
| **Filesystem** | ⚠️ Selbst bauen | ✅ `ls`, `read_file`, `write_file` automatisch |
| **Sub-Agenten** | ⚠️ Aufwändig (M30-Pattern) | ✅ `subagents=[...]` Parameter |
| **Debugging** | ✅ Direkt (man sieht jeden Schritt) | ⚠️ Erfordert LangGraph-Kenntnisse |
| **API-Stabilität** | ✅ LangGraph 1.0 — produktionserprobt | ✅ DeepAgents 0.5.0 erreicht — stabiler als 0.4.x, API-Änderungen seltener |
| **Use Case** | Custom Workflows, präzise Kontrolle | Autonome Long-Running Tasks |

**Faustregel:**
- ✅ LangGraph manuell: wenn jeder Schritt exakt kontrolliert werden muss
- ✅ DeepAgents: wenn ein autonomer Agent eine komplexe Aufgaben lösen soll

<p><font color='darkblue' size="4">🚨 <b>Achtung</b></font></p>

DeepAgents setzt LangGraph-Grundkenntnisse (StateGraph bis Supervisor Pattern) voraus. Die Harness bricht — und dann braucht man das Fundament, um zu debuggen. Das Erlernen aller vorherigen Module ist keine Vorbereitung auf die Abkürzung. Es ist die Voraussetzung dafür, die Abkürzung sinnvoll einzusetzen.

---

**Zur API-Stabilität — eine sachliche Einschätzung:**

DeepAgents 0.4.x ist eine junge Bibliothek. Das `subagents=`-Interface wurde in diesem
Notebook bereits korrigiert: Die Schnittstelle erwartet ein **Dict**, nicht einen
`CompiledStateGraph` — ein Breaking Change, der in keiner offiziellen Migrationsanleitung
dokumentiert war.

Das ist kein Einzelfall für frühe 0.x-Releases. Konkrete Konsequenz für den produktiven Einsatz:

```python
**Empfehlung: Version pinnen — in diesem Notebook bereits gesetzt:**
pip install deepagents==0.6.12   # Setup-Zelle oben verwendet diese Version
```

Für **experimentelle Projekte und Prototypen** ist DeepAgents gut geeignet.
Für **Production-Deployments** gilt: LangGraph direkt ist stabiler,
dokumentierter und hat ein größeres Ökosystem. DeepAgents lohnt sich zu beobachten —
ab v1.0 mit stabilem API ändert sich die Einschätzung. Mit 0.5.0 ist ein wichtiger Stabilitäts-Meilenstein erreicht.

In [ ]:
#@markdown   <p><font size="4" color='green'>   Entscheidungsbaum</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    A(["Neue Agenten-Aufgabe"])
    B{"Präzise Kontrolle\nüber jeden Schritt?"}
    C{"Autonome Aufgaben\nmit vielen Teilschritten?"}
    D["LangGraph manuell\nM13-M14-Pattern"]
    E["DeepAgents\ncreate_deep_agent()"]
    F{"Lernzweck oder\nProduction?"}
    G["LangGraph\n(Kurs M13-M32)"]
    H["DeepAgents\n(M33)"]
    A --> B
    B -->|Ja| D
    B -->|Nein| C
    C -->|Ja| E
    C -->|Nein| F
    F -->|Lernen| G
    F -->|Production| H
    style A fill:#E65100,color:#fff
    style D fill:#1565C0,color:#fff
    style E fill:#2E7D32,color:#fff
    style G fill:#37474F,color:#fff
    style H fill:#37474F,color:#fff
'''
mermaid(diagram, width=500)

<p><font color='black' size="5">
Optionaler Kontext: Warum jetzt?
</font></p>

2024/2025 sind autonome Agenten-Systeme mit Harness-Pattern viral gegangen.
LangChain hat in einem Blog-Post die gemeinsame Architektur analysiert:

| System | Entwickler | Planning | Filesystem | Sub-Agenten |
|--------|-----------|----------|------------|-------------|
| **Claude Code** | Anthropic | ✅ Todo-Liste | ✅ read/write/edit | ✅ Sub-Tasks |
| **Manus** | Monica | ✅ Planung | ✅ Dateiverwaltung | ✅ Delegation |
| **Deep Research** | OpenAI | ✅ Search-Plan | ✅ Context-Offloading | ⚠️ Begrenzt |
| **DeepAgents** | LangChain | ✅ `write_todos` | ✅ ls/read/write | ✅ `subagents=[]` |

> *"Applications like Deep Research, Manus, and Claude Code have gotten around the*
> *limitation of shallow agents by implementing a combination of four things:*
> *a planning tool, sub agents, access to a file system, and a detailed prompt."*
> — LangChain Blog

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf und soll im Kurs auch als Lernunterstützung genutzt werden — z. B. Gemini in Google Colab, um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.

**Grundlagen**

Vergleiche DeepAgents und LangGraph direkt: Implementiere dieselbe Aufgaben mit beiden Frameworks und dokumentiere Lines of Code, Latenz und Qualität.

**✅ Erledigt wenn:** Tabelle zeigt für beide Implementierungen LOC, Latenz und Qualitätseinschätzung — Empfehlung ist mit konkreten Zahlen begründet.

In [ ]:
# Vertiefung: DeepAgents vs. LangGraph
import time

aufgabe = 'Analysiere und fasse zusammen: ...'

# DeepAgents-Implementierung
t0 = time.time()
# result_da = deepagent.run(aufgabe)
latenz_da = time.time() - t0

# LangGraph-Implementierung
t0 = time.time()
# result_lg = langgraph_app.invoke({'input': Aufgaben}, config=run_cfg)
latenz_lg = time.time() - t0

vergleich = [
    ['DeepAgents', 0,  latenz_da, ''],  # LOC eintragen
    ['LangGraph',  0,  latenz_lg, ''],
]
# mprint(pd.DataFrame(vergleich, columns=['Framework','LOC','Latenz','Qualität']).to_markdown())

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [LangGraph](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)
- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)
- [Modellsteuerung](https://editor.p5js.org/ralf.bendig.rb/full/um423ggnD)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [LangGraph - Überblick](https://ralf-42.github.io/Agenten/05-frameworks/langgraph.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
